In [25]:
# Loading stuff

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [26]:
# Loading all the datasets

# Train dataset
train_dataset = pd.read_csv('train_radiomics_hipocamp.csv')

# Test dataset
test_dataset = pd.read_csv('test_radiomics_hipocamp.csv')

# Control dataset
control_dataset = pd.read_csv('train_radiomics_occipital_CONTROL.csv')

In [ ]:
# Exploring the train dataset

train_dataset.head()

In [ ]:
train_dataset.columns

In [ ]:
train_dataset.info(verbose=True, show_counts=True)

In [ ]:
# Getting all columns whose value is an object
obj_col = train_dataset.select_dtypes(include='object').columns

# For each of those, checking the number of unique values
# Storing in a list the columns that have the same number of unique values as the number of rows
# This means that the column is a unique identifier and probably should be dropped
unique_cols = []
for col in obj_col:
    if train_dataset[col].nunique() == 305:
        unique_cols.append(col)

# Creating and displaying a dataframe containing only the columns stored in the above list
unique_df = train_dataset[unique_cols]
unique_df.head()


In [31]:
# It looks like all the columns in the unique_df dataframe are unique identifiers that don't provide any useful information

# Dropping the columns stored in the unique_cols list
train_dataset.drop(unique_cols, axis=1, inplace=True)
# Dropping them from the test dataset as well
test_dataset.drop(unique_cols, axis=1, inplace=True)

In [ ]:
# Listing columns by type
dict = {}
for col in train_dataset.columns:
    # get the type
    dtype = str(train_dataset[col].dtype)
    if dtype in dict:
        dict[dtype] += 1
    else:
        dict[dtype] = 1

print(dict)

In [ ]:
# Dropping all collumns with only one unique value

only_one_unique_val_col = 0
for col in train_dataset.columns:
    if train_dataset[col].nunique() == 1:
        only_one_unique_val_col += 1
        train_dataset.drop(col, axis=1, inplace=True)
        # Dropping them from the test dataset as well
        test_dataset.drop(col, axis=1, inplace=True)

print("dropped " + str(only_one_unique_val_col) + " columns")
        

In [ ]:
# Listing columns by type again
dict = {}
for col in train_dataset.columns:
    # get the type
    dtype = str(train_dataset[col].dtype)
    if dtype in dict:
        dict[dtype] += 1
    else:
        dict[dtype] = 1

print(dict)

In [ ]:
# Checking all unique values in the Transition column
train_dataset['Transition'].unique()

In [ ]:
# Using encoding to transform the categorical columns into numerical columns
replace_map = {'Transition': {'CN-CN': 0, 'AD-AD': 1, 'CN-MCI': 2, 'MCI-AD': 3, 'MCI-MCI': 4}}
train_dataset.replace(replace_map, inplace=True)
test_dataset.replace(replace_map, inplace=True)
train_dataset.head()

In [ ]:
# Checking for columns with repeated information
# We will check the correlation between the columns
# If the correlation is 1, this means that the columns are the same
# We will store the columns that have a correlation of 1 in a list

corr_matrix = train_dataset.corr().abs()
# Getting all the columns that have a correlation of 1 with another column
corr_cols = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) == 1:
            corr_cols.append(corr_matrix.columns[i])

print(corr_cols)

In [38]:
# Dropping the columns stored in the corr_cols list
train_dataset.drop(corr_cols, axis=1, inplace=True)
test_dataset.drop(corr_cols, axis=1, inplace=True)

In [ ]:
# Listing columns by type
dict = {}
for col in train_dataset.columns:
    # get the type
    dtype = str(train_dataset[col].dtype)
    if dtype in dict:
        dict[dtype] += 1
    else:
        dict[dtype] = 1

print(dict)

In [ ]:
# Checking each element correlation with the target column
corr_matrix = train_dataset.corr()
corr_matrix['Transition'].sort_values(ascending=False)

In [62]:
# Running a Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

rfc = RandomForestClassifier(n_estimators=100, random_state=123)
rfc.fit(X_train, y_train)
rfc_pred = rfc.predict(X_test)

In [ ]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, rfc_pred))

In [ ]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, rfc_pred))

In [ ]:
# Printing f1 score
from sklearn.metrics import f1_score
print(f1_score(y_test, rfc_pred, average='weighted'))

In [ ]:
# Using the model to complete the test dataset
test_pred = rfc.predict(test_dataset)
test_dataset['Transition'] = test_pred
test_dataset.head()


In [43]:
# Dropping all columns but the Transition column
test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

# Creating a RowId column to store the index, starting from 1
test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

# Placing the RowId column in the first position
cols = test_dataset.columns.tolist()
cols = cols[-1:] + cols[:-1]
test_dataset = test_dataset[cols]

In [ ]:
test_dataset.head()

In [ ]:
# Transforming the Transition column back to its original values
replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
test_dataset.replace(replace_map, inplace=True)
test_dataset.head()

In [46]:
# Saving the test dataset to a csv file
test_dataset.to_csv('test_predictions.csv', index=False)